# Threshold sensitivity and excluded test images by class

In [1]:
import os, json
import numpy as np
import pandas as pd

DATA_DIR = "."          # folder with the input files
OUT_DIR = "."           # folder for the two output CSV files
K = 45                  # number of classes

PRED_FILES = {
    "ViT-B/16": "vit_b16-1_test_preds.csv",
    "Improved Swin": "swint_dual_test_preds.csv",
    "MobileNetV3 Large": "mobilenetv3large_test_preds.csv",
    "ResNet50": "resnet50_test_preds.csv",
    "DenseNet121": "densenet121_test_preds.csv",
}
THRESHOLDS_RECOMPUTED = [0.90, 0.88, 0.86, 0.84]   # computable from embed_test_vs_pool.csv

# Counts copied from the output of the near duplicate notebook (not recomputed here)
TEST_COUNT_FROM_NOTEBOOK = {0.90: 131, 0.88: 202, 0.86: 292, 0.84: 380, 0.82: 502}
TRAIN_POOL_FROM_NOTEBOOK = {0.84: (2696, 20.8), 0.90: (911, 7.0), 0.92: (584, 4.5),
                            0.94: (313, 2.4), 0.96: (122, 0.9), 0.98: (34, 0.3)}

p = lambda name: os.path.join(DATA_DIR, name)

In [2]:
emb = pd.read_csv(p("embed_test_vs_pool.csv"))
emb["fp"] = emb["test_file"].map(os.path.normpath)
print(len(emb), "rows;", emb["test_file"].nunique(), "unique test images; min cosine", emb["cosine"].min())

preds = {k: pd.read_csv(p(v)) for k, v in PRED_FILES.items()}
ref = preds["ViT-B/16"].copy()
for k, d in preds.items():          # all files must list the same images in the same order
    assert (d["filepath"].values == ref["filepath"].values).all(), k
    assert (d["y_true"].values == ref["y_true"].values).all(), k
ref["fp"] = ref["filepath"].map(os.path.normpath)
ref["cls"] = ref["filepath"].str.split("/").str[1].str.replace(" architecture", "", regex=False)
print(len(ref), "test images;", ref["cls"].nunique(), "classes")

500 rows; 500 unique test images; min cosine 0.8204
1438 test images; 45 classes


In [3]:
def acc_macro_f1(y_true, y_pred, k=K):
    """Accuracy and macro F1 from the k x k confusion matrix.
    A label absent from both y_true and y_pred is left out of the mean;
    a label with true images but no predictions (or the reverse) counts as zero.
    Same definition as the paired statistics notebook."""
    cm = np.bincount(y_true * k + y_pred, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    pr, tr = cm.sum(0).astype(float), cm.sum(1).astype(float)
    present = (pr + tr) > 0
    f1 = np.where(present, 2 * tp / np.where(present, pr + tr, 1), 0.0)
    return tp.sum() / cm.sum(), f1[present].mean()

In [4]:
def row_for(threshold):
    if threshold is None:
        mask = np.ones(len(ref), bool)
    else:
        flagged = set(emb.loc[emb["cosine"] >= threshold, "fp"])
        mask = ~ref["fp"].isin(flagged).values
    r = {"cosine_threshold": "none (original test set)" if threshold is None else threshold,
         "test_images_excluded": int((~mask).sum()),
         "test_images_remaining": int(mask.sum())}
    for k, d in preds.items():
        a, f = acc_macro_f1(d["y_true"].values[mask], d["y_pred"].values[mask])
        r[f"{k}_accuracy"], r[f"{k}_macro_F1"] = round(a, 4), round(f, 4)
    return r, mask

rows = [row_for(None)[0]] + [row_for(t)[0] for t in THRESHOLDS_RECOMPUTED]
table = pd.DataFrame(rows)

# Cross checks against numbers already in the manuscript
r84 = table[table["cosine_threshold"] == 0.84].iloc[0]
assert r84["test_images_excluded"] == 380, "380 excluded images expected at 0.84"
assert r84["test_images_remaining"] == 1058
assert abs(r84["ViT-B/16_macro_F1"] - 0.8133) < 5e-5, "ViT-B/16 macro F1 on the cleaned test set should be 0.8133"
for t in THRESHOLDS_RECOMPUTED:
    got = int(table[table["cosine_threshold"] == t]["test_images_excluded"].iloc[0])
    assert got == TEST_COUNT_FROM_NOTEBOOK[t], (t, got, TEST_COUNT_FROM_NOTEBOOK[t])

# ranking by macro F1 at every threshold
f1_cols = [c for c in table.columns if c.endswith("_macro_F1")]
for _, r in table.iterrows():
    order = sorted(f1_cols, key=lambda c: -r[c])
    print(r["cosine_threshold"], " > ".join(c.replace("_macro_F1", "") for c in order))
table

none (original test set) ViT-B/16 > Improved Swin > MobileNetV3 Large > ResNet50 > DenseNet121
0.9 ViT-B/16 > Improved Swin > MobileNetV3 Large > ResNet50 > DenseNet121
0.88 ViT-B/16 > Improved Swin > MobileNetV3 Large > ResNet50 > DenseNet121
0.86 ViT-B/16 > Improved Swin > MobileNetV3 Large > ResNet50 > DenseNet121
0.84 ViT-B/16 > Improved Swin > MobileNetV3 Large > ResNet50 > DenseNet121


,cosine_threshold,test_images_excluded,test_images_remaining,ViT-B/16_accuracy,ViT-B/16_macro_F1,Improved Swin_accuracy,Improved Swin_macro_F1,MobileNetV3 Large_accuracy,MobileNetV3 Large_macro_F1,ResNet50_accuracy,ResNet50_macro_F1,DenseNet121_accuracy,DenseNet121_macro_F1
0,none (original test set),0,1438,0.8470,0.8511,0.8352,0.8347,0.8234,0.8232,0.8122,0.8058,0.6912,0.6956
1,0.9,131,1307,0.8332,0.8402,0.8217,0.8234,0.8095,0.8106,0.7965,0.7913,0.6741,0.6803
2,0.88,202,1236,0.8236,0.8333,0.8123,0.8184,0.7994,0.8042,0.7856,0.7841,0.6594,0.6678
3,0.86,292,1146,0.8133,0.8223,0.8019,0.8086,0.7871,0.7914,0.7723,0.7710,0.6457,0.6535
4,0.84,380,1058,0.8025,0.8133,0.7921,0.8023,0.7769,0.7822,0.7618,0.7622,0.6323,0.6427


In [5]:
# Add the rows that are only available as counts from the near duplicate notebook
extra = [{"cosine_threshold": t,
          "test_images_excluded": TEST_COUNT_FROM_NOTEBOOK.get(t, ""),
          "test_images_remaining": (len(ref) - TEST_COUNT_FROM_NOTEBOOK[t]) if t in TEST_COUNT_FROM_NOTEBOOK else ""}
         for t in [0.82, 0.92, 0.94, 0.96, 0.98]]
full = pd.concat([table, pd.DataFrame(extra)], ignore_index=True)
full["train_pool_images_with_neighbor_in_other_fold"] = full["cosine_threshold"].map(
    lambda t: TRAIN_POOL_FROM_NOTEBOOK[t][0] if t in TRAIN_POOL_FROM_NOTEBOOK else None)
full["train_pool_share_percent"] = full["cosine_threshold"].map(
    lambda t: TRAIN_POOL_FROM_NOTEBOOK[t][1] if t in TRAIN_POOL_FROM_NOTEBOOK else None)

full.to_csv(os.path.join(OUT_DIR, "threshold_sensitivity.csv"), index=False, encoding="utf8")
full

,cosine_threshold,test_images_excluded,test_images_remaining,ViT-B/16_accuracy,ViT-B/16_macro_F1,Improved Swin_accuracy,Improved Swin_macro_F1,MobileNetV3 Large_accuracy,MobileNetV3 Large_macro_F1,ResNet50_accuracy,ResNet50_macro_F1,DenseNet121_accuracy,DenseNet121_macro_F1,train_pool_images_with_neighbor_in_other_fold,train_pool_share_percent
0,none (original test set),0,1438,0.8470,0.8511,0.8352,0.8347,0.8234,0.8232,0.8122,0.8058,0.6912,0.6956,NaN,NaN
1,0.9,131,1307,0.8332,0.8402,0.8217,0.8234,0.8095,0.8106,0.7965,0.7913,0.6741,0.6803,911.0,7.0
2,0.88,202,1236,0.8236,0.8333,0.8123,0.8184,0.7994,0.8042,0.7856,0.7841,0.6594,0.6678,NaN,NaN
3,0.86,292,1146,0.8133,0.8223,0.8019,0.8086,0.7871,0.7914,0.7723,0.7710,0.6457,0.6535,NaN,NaN
4,0.84,380,1058,0.8025,0.8133,0.7921,0.8023,0.7769,0.7822,0.7618,0.7622,0.6323,0.6427,2696.0,20.8
5,0.82,502,936,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.92,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,584.0,4.5
7,0.94,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,313.0,2.4
8,0.96,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122.0,0.9
9,0.98,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.0,0.3


In [6]:
# Excluded test images by class at cosine 0.84
_, mask84 = row_for(0.84)
ref["excluded"] = ~mask84

# Optional check: same images as flagged_test_files.json?
with open(p("flagged_test_files.json")) as f:
    flagged_json = set(os.path.normpath(x) for x in json.load(f))
print("same set as flagged_test_files.json:", set(ref.loc[ref["excluded"], "fp"]) == flagged_json)

g = ref.groupby("cls")["excluded"].agg(["count", "sum"]).rename(
    columns={"count": "test_images_original", "sum": "test_images_excluded"})
g["test_images_retained"] = g["test_images_original"] - g["test_images_excluded"]
g["share_excluded_percent"] = (100 * g["test_images_excluded"] / g["test_images_original"]).round(1)
summary = []
for k, d in preds.items():
    correct = pd.Series((d["y_true"] == d["y_pred"]).values, index=ref.index)
    g[f"{k}_acc_excluded"] = correct[ref["excluded"]].groupby(ref["cls"][ref["excluded"]]).mean().round(4)
    g[f"{k}_acc_retained"] = correct[~ref["excluded"]].groupby(ref["cls"][~ref["excluded"]]).mean().round(4)
    # expected accuracy on the excluded images if they were as hard as retained images of the same class
    per_class_ret = correct[~ref["excluded"]].groupby(ref["cls"][~ref["excluded"]]).mean()
    w = ref.loc[ref["excluded"], "cls"].value_counts()
    expected = (per_class_ret.reindex(w.index).fillna(per_class_ret.mean()) * w).sum() / w.sum()
    summary.append({"model": k, "acc_excluded": round(correct[ref["excluded"]].mean(), 4),
                    "acc_retained": round(correct[~ref["excluded"]].mean(), 4),
                    "expected_acc_excluded_given_class_mix": round(expected, 4)})
g = g.sort_values("test_images_excluded", ascending=False)
g.to_csv(os.path.join(OUT_DIR, "excluded_by_class.csv"), encoding="utf8")
print("classes with no excluded image:", list(g.index[g["test_images_excluded"] == 0]))
pd.DataFrame(summary)

same set as flagged_test_files.json: True
classes with no excluded image: ['Jain']


,model,acc_excluded,acc_retained,expected_acc_excluded_given_class_mix
0,ViT-B/16,0.9711,0.8025,0.8317
1,Improved Swin,0.9553,0.7921,0.8292
2,MobileNetV3 Large,0.9526,0.7769,0.8137
3,ResNet50,0.9526,0.7618,0.7895
4,DenseNet121,0.8553,0.6323,0.6715
